# Tome C2, Chapter 9: Algebraic Algorithms (Python companion)

Exercises first, solutions at the end. We implement the tower of Chapter 9 on exact arithmetic: Karatsuba and Toom-3 against the schoolbook method, an FFT in the ring $\mathbb Z/(2^K+1)$ where the root of unity is $2$, a number-theoretic transform modulo $998\,244\,353$ as the multiplication engine, Newton inversion and fast division of power series, multipoint evaluation and interpolation with subproduct trees, block inversion of matrices, and Fiduccia's algorithm for the $10^{18}$th term of a linear recurrence.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time
import numpy as np
import matplotlib.pyplot as plt
random.seed(9); rng = np.random.default_rng(9)

## Exercises

**P9.1 (★★) Splitting.** Implement schoolbook, Karatsuba and Toom-3 multiplication of integer polynomials (Theorem 9.1, Exercise 9.2), check them against each other, count coefficient multiplications, and time them. How does Python's own `int` multiplication scale?

**P9.2 (★★) Shifts instead of multiplications.** In $R_K=\mathbb Z/(2^K+1)$, implement the FFT of length $2K$ with root $\omega=2$ (Lemma 9.4), using only shifts, additions and reductions, and check the convolution theorem (Proposition 9.3).

**P9.3 (★★) Newton.** Implement an NTT modulo $p=998\,244\,353=119\cdot2^{23}+1$, then power series inversion by Newton's iteration (Theorem 9.5) and fast division (Corollary 9.6). Compare with the quadratic methods.

**P9.4 (★★★) Many points.** Implement the subproduct tree, multipoint evaluation (Theorem 9.7) and fast interpolation (Theorem 9.8) modulo $p$; check against Horner's rule and find the crossover.

**P9.5 (★★) Matrices.** Implement Strassen's recursion and the Schur complement inversion of Theorem 9.9 with `numpy`, check them, and compare with `numpy.linalg.inv`.

**P9.6 (★) Recurrences.** Compute the $10^{18}$th term modulo $p$ of a random linear recurrence of order $d$ by Fiduccia's method (Theorem 9.10) and by powering the companion matrix, for $d$ up to a few hundred.

## Solutions

### P9.1 Splitting
Polynomials are lists of integer coefficients, lowest degree first. Each function counts the products of coefficients it performs, in a global counter.

In [ ]:
MULS = [0]
def add(a, b):
    if len(a) < len(b): a, b = b, a
    return [x + (b[i] if i < len(b) else 0) for i, x in enumerate(a)]
def sub(a, b):
    return add(a, [-x for x in b])

def school(a, b):
    if not a or not b: return []
    c = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            c[i + j] += x * y
    MULS[0] += len(a) * len(b)
    return c

def karatsuba(a, b, cut=1):
    n = max(len(a), len(b))
    if n <= cut or min(len(a), len(b)) == 0: return school(a, b)
    h = (n + 1) // 2
    a0, a1, b0, b1 = a[:h], a[h:], b[:h], b[h:]
    p0, p2 = karatsuba(a0, b0, cut), karatsuba(a1, b1, cut)
    p1 = sub(sub(karatsuba(add(a0, a1), add(b0, b1), cut), p0), p2)
    c = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(p0): c[i] += x
    for i, x in enumerate(p1): c[i + h] += x
    for i, x in enumerate(p2): c[i + 2 * h] += x
    return c

def toom3(a, b, cut=2):
    n = max(len(a), len(b))
    if n <= cut or min(len(a), len(b)) == 0: return school(a, b)
    h = (n + 2) // 3
    A = [a[i * h:(i + 1) * h] for i in range(3)]; B = [b[i * h:(i + 1) * h] for i in range(3)]
    def ev(P, x):  # P0 + x P1 + x^2 P2, x small integer
        return add(add(P[0], [x * t for t in P[1]]), [x * x * t for t in P[2]])
    c0 = toom3(A[0], B[0], cut); c4 = toom3(A[2], B[2], cut)
    c1 = toom3(ev(A, 1), ev(B, 1), cut); cm1 = toom3(ev(A, -1), ev(B, -1), cut); c2 = toom3(ev(A, 2), ev(B, 2), cut)
    L = 2 * h + 1
    pad = lambda p: p + [0] * (L - len(p))
    c0, c4, c1, cm1, c2 = map(pad, (c0, c4, c1, cm1, c2))
    s = [(x + y) // 2 for x, y in zip(c1, cm1)]; t = [(x - y) // 2 for x, y in zip(c1, cm1)]
    C2 = [s_ - x - y for s_, x, y in zip(s, c0, c4)]
    C3 = [(v - x - 4 * w - 16 * y - 2 * u) // 6 for v, x, w, y, u in zip(c2, c0, C2, c4, t)]
    C1 = [u - w for u, w in zip(t, C3)]
    c = [0] * (len(a) + len(b) - 1)
    for k, Ck in enumerate((c0, C1, C2, C3, c4)):
        for i, x in enumerate(Ck):
            if k * h + i < len(c): c[k * h + i] += x
    return c

for n in (1, 2, 5, 17, 64, 100):
    a = [random.randint(-99, 99) for _ in range(n)]; b = [random.randint(-99, 99) for _ in range(n)]
    assert school(a, b) == karatsuba(a, b) == toom3(a, b), n
print("all three agree")
print(" n    schoolbook   Karatsuba    Toom-3     n^1.585   n^1.465")
for r in range(4, 11):
    n = 2**r if r % 2 == 0 else 3**((r + 1) // 2)
    a = [random.randint(-9, 9) for _ in range(n)]; b = [random.randint(-9, 9) for _ in range(n)]
    out = []
    for f in (school, karatsuba, toom3):
        MULS[0] = 0; f(a, b); out.append(MULS[0])
    print(f"{n:5d} {out[0]:11d} {out[1]:11d} {out[2]:10d} {n**math.log2(3):10.0f} {n**math.log(5, 3):9.0f}")

For $n=2^r$ Karatsuba makes exactly $3^r=n^{\log_23}$ products, and Toom-3 makes $5^s=n^{\log_35}$ for $n=3^s$ (with the cut-off at size $2$ adding a constant factor). The counts are not the whole story: additions and bookkeeping cost time too, so we switch to the schoolbook method below $32$ coefficients, and the timings below show the splitting methods paying off from about a hundred coefficients on.

In [ ]:
sizes = [32, 64, 128, 256, 512, 1024]
times = {"schoolbook": [], "Karatsuba (cut 32)": [], "Toom-3 (cut 32)": []}
for n in sizes:
    a = [random.randint(-10**6, 10**6) for _ in range(n)]; b = [random.randint(-10**6, 10**6) for _ in range(n)]
    for name, f in (("schoolbook", school), ("Karatsuba (cut 32)", lambda x, y: karatsuba(x, y, 32)), ("Toom-3 (cut 32)", lambda x, y: toom3(x, y, 32))):
        t0 = time.perf_counter(); f(a, b); times[name].append(time.perf_counter() - t0)
plt.figure(figsize=(6, 3.3))
for name, ts in times.items(): plt.loglog(sizes, ts, "o-", label=name)
plt.xlabel("n (coefficients)"); plt.ylabel("seconds"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()
for name, ts in times.items():
    print(f"{name:20s} slope {np.polyfit(np.log(sizes[2:]), np.log(ts[2:]), 1)[0]:.2f}")

Python's integers use the schoolbook method below about $70$ machine digits of $30$ bits and Karatsuba above: the measured exponent of `int` multiplication should approach $\log_23\approx1.585$.

In [ ]:
bits = [2**k for k in range(14, 22)]; ts = []
for nb in bits:
    x = random.getrandbits(nb) | 1; y = random.getrandbits(nb) | 1
    reps = max(1, 2**21 // nb); t0 = time.perf_counter()
    for _ in range(reps): x * y
    ts.append((time.perf_counter() - t0) / reps)
print(f"measured exponent of int multiplication: {np.polyfit(np.log(bits), np.log(ts), 1)[0]:.3f}   (log2 3 = {math.log2(3):.3f})")

### P9.2 Shifts instead of multiplications
Elements of $R_K$ are Python integers in $[0,2^K]$. Multiplying by $2^s$ is a shift followed by a reduction modulo $2^K+1$, which uses $2^K\equiv-1$: split into high and low parts and subtract.

In [ ]:
def reduce_fermat(x, K):
    M = (1 << K) + 1
    while x >> (K + 1):
        x = (x & ((1 << K) - 1)) - (x >> K)
        if x < 0: x += M
    return x % M

def mul_pow2(x, s, K):
    return reduce_fermat(x << (s % (2 * K)), K)

def fft_fermat(a, K, inverse=False):
    # length n = len(a) divides 2K; root of unity w = 2^(2K/n), inverse w^-1 = 2^(2K - 2K/n)
    n = len(a)
    if n == 1: return a[:]
    step = 2 * K // n
    E = fft_fermat(a[0::2], K, inverse); O = fft_fermat(a[1::2], K, inverse)
    M = (1 << K) + 1; out = [0] * n
    for j in range(n // 2):
        e = (j * step) if inverse else (-j * step)
        t = mul_pow2(O[j], e % (2 * K), K)
        out[j] = (E[j] + t) % M; out[j + n // 2] = (E[j] - t) % M
    return out

K = 64; n = 2 * K; M = (1 << K) + 1
a = [random.randrange(M) for _ in range(n)]; b = [random.randrange(M) for _ in range(n)]
A, B = fft_fermat(a, K), fft_fermat(b, K)
conv = fft_fermat([x * y % M for x, y in zip(A, B)], K, inverse=True)
ninv = pow(n, -1, M); conv = [x * ninv % M for x in conv]
direct = [sum(a[l] * b[(k - l) % n] for l in range(n)) % M for k in range(n)]
print("convolution theorem in Z/(2^64+1) with root 2:", conv == direct)
print("principal root check: sums of 2^(jk) for 0<k<2K all zero:",
      all(sum(pow(2, j * k, M) for j in range(2 * K)) % M == 0 for k in range(1, 2 * K)))

Every multiplication inside the transform was a shift and a subtraction; only the pointwise products `x * y % M` are genuine products, and those are what Schönhage and Strassen do recursively.

### P9.3 Newton
The multiplication engine: an iterative NTT modulo $p=998\,244\,353$, whose multiplicative group has the element $3$ of order $p-1=119\cdot2^{23}$, so $3^{119\cdot2^{23-r}}$ is a principal $2^r$th root of unity (Tome 4, Theorem 7.11). We vectorise the butterflies with `numpy` (int64 products of numbers below $2^{30}$ stay below $2^{60}$).

In [ ]:
P = 998244353; G = 3
def ntt(a, invert=False):
    a = np.array(a, dtype=np.int64) % P; n = len(a)
    j = 0; rev = np.zeros(n, dtype=np.int64)
    bits = n.bit_length() - 1
    rev = np.array([int(format(i, f"0{bits}b")[::-1], 2) if bits else 0 for i in range(n)])
    a = a[rev]
    length = 2
    while length <= n:
        w = pow(G, (P - 1) // length, P)
        if invert: w = pow(w, P - 2, P)
        half = length // 2
        ws = np.ones(half, dtype=np.int64)
        for k in range(1, half): ws[k] = ws[k - 1] * w % P
        a = a.reshape(-1, length)
        u = a[:, :half].copy(); v = a[:, half:] * ws % P
        a[:, :half] = (u + v) % P; a[:, half:] = (u - v) % P
        a = a.reshape(-1); length *= 2
    if invert: a = a * pow(n, P - 2, P) % P
    return a

def pmul(a, b):
    # product of polynomials modulo P
    if len(a) == 0 or len(b) == 0: return np.zeros(0, dtype=np.int64)
    m = len(a) + len(b) - 1
    if min(len(a), len(b)) <= 32:
        return np.array(np.convolve(np.array(a, dtype=object), np.array(b, dtype=object)) % P, dtype=np.int64)
    n = 1 << (m - 1).bit_length()
    fa = ntt(np.concatenate([a, np.zeros(n - len(a), dtype=np.int64)]))
    fb = ntt(np.concatenate([b, np.zeros(n - len(b), dtype=np.int64)]))
    return ntt(fa * fb % P, invert=True)[:m]

x = rng.integers(0, P, 300); y = rng.integers(0, P, 200)
print("NTT product matches exact convolution:", np.array_equal(pmul(x, y), np.array(np.convolve(x.astype(object), y.astype(object)) % P, dtype=np.int64)))

In [ ]:
def inverse_series(f, n):
    # first n coefficients of 1/f modulo P, by Newton: g <- g (2 - f g)
    g = np.array([pow(int(f[0]), P - 2, P)], dtype=np.int64); k = 1
    while k < n:
        k = min(2 * k, n)
        fg = pmul(f[:k], g)[:k]
        two_minus = (-fg) % P; two_minus[0] = (two_minus[0] + 2) % P
        g = pmul(g, two_minus)[:k]
    return g

def inverse_naive(f, n):
    g = [0] * n; inv0 = pow(int(f[0]), P - 2, P); g[0] = inv0
    for k in range(1, n):
        s = sum(int(f[i]) * g[k - i] for i in range(1, min(k, len(f) - 1) + 1)) % P
        g[k] = (-s) * inv0 % P
    return np.array(g, dtype=np.int64)

f = rng.integers(0, P, 4096); f[0] = 7
for n in (256, 1024, 4096):
    t0 = time.perf_counter(); g1 = inverse_series(f, n); t1 = time.perf_counter() - t0
    t0 = time.perf_counter(); g2 = inverse_naive(f, n) if n <= 1024 else None; t2 = time.perf_counter() - t0
    check = pmul(f[:n], g1)[:n]
    ok = check[0] == 1 and not check[1:].any()
    print(f"n={n:5d}: Newton {t1 * 1000:7.1f} ms (f*g = 1 mod x^n: {ok})" + (f",  coefficient by coefficient {t2 * 1000:8.1f} ms, same: {np.array_equal(g1, g2)}" if g2 is not None else ""))

In [ ]:
def divmod_poly(a, b):
    # a, b coefficient arrays (lowest first), b monic or with invertible leading coefficient
    n, m = len(a) - 1, len(b) - 1
    if n < m: return np.zeros(1, dtype=np.int64), a
    ra, rb = a[::-1], b[::-1]
    q_rev = pmul(ra, inverse_series(rb, n - m + 1))[: n - m + 1]
    q = q_rev[::-1]
    r = (a - np.concatenate([pmul(q, b), np.zeros(0, dtype=np.int64)])[: len(a)]) % P
    return q, r[:m]

a = rng.integers(0, P, 5001); b = rng.integers(0, P, 2001); b[-1] = 1
q, r = divmod_poly(a, b)
back = (np.concatenate([pmul(q, b), np.zeros(1, dtype=np.int64)])[:len(a)] + np.concatenate([r, np.zeros(len(a) - len(r), dtype=np.int64)])) % P
print("a = q b + r with deg r < deg b:", np.array_equal(back, a % P), len(r) == len(b) - 1)

### P9.4 Many points
The subproduct tree is a list of levels; level 0 holds the linear polynomials $x-u_i$.

In [ ]:
def subproduct_tree(us):
    level = [np.array([(-u) % P, 1], dtype=np.int64) for u in us]
    tree = [level]
    while len(level) > 1:
        level = [pmul(level[2 * i], level[2 * i + 1]) for i in range(len(level) // 2)]
        tree.append(level)
    return tree

def poly_mod(f, m):
    if len(f) < len(m): return f
    return divmod_poly(f, m)[1]

def multipoint(f, tree):
    rems = [poly_mod(f, tree[-1][0])]
    for level in reversed(tree[:-1]):
        rems = [poly_mod(rems[i // 2], level[i]) for i in range(len(level))]
    return np.array([int(r[0]) if len(r) else 0 for r in rems], dtype=np.int64)

def horner_all(f, us):
    out = []
    for u in us:
        v = 0
        for c in reversed(f.tolist()): v = (v * u + c) % P
        out.append(v)
    return np.array(out, dtype=np.int64)

def interpolate(us, vs, tree):
    m = tree[-1][0]
    dm = np.array([(k * int(c)) % P for k, c in enumerate(m)][1:], dtype=np.int64)   # derivative of m
    w = multipoint(dm, tree)
    cs = [int(v) * pow(int(x), P - 2, P) % P for v, x in zip(vs, w)]
    level = [np.array([c], dtype=np.int64) for c in cs]
    for k in range(len(tree) - 1):
        ms = tree[k]
        level = [(np.concatenate([pmul(level[2 * i], ms[2 * i + 1]), [0]])[: len(ms[2 * i]) + len(ms[2 * i + 1]) - 1]
                  + np.concatenate([pmul(level[2 * i + 1], ms[2 * i]), [0]])[: len(ms[2 * i]) + len(ms[2 * i + 1]) - 1]) % P
                 for i in range(len(level) // 2)]
    return level[0][: len(us)]

n = 512
us = rng.choice(P, n, replace=False); f = rng.integers(0, P, n)
tree = subproduct_tree(us)
vals = multipoint(f, tree)
print("multipoint = Horner:", np.array_equal(vals, horner_all(f, us)))
g = interpolate(us, vals, tree)
print("interpolation recovers f:", np.array_equal(g % P, f % P))

In [ ]:
rows = []
for k in range(6, 13):
    n = 2**k; us = rng.choice(P, n, replace=False); f = rng.integers(0, P, n)
    t0 = time.perf_counter(); tree = subproduct_tree(us); v = multipoint(f, tree); t_fast = time.perf_counter() - t0
    if n <= 2048:
        t0 = time.perf_counter(); horner_all(f, us); t_h = time.perf_counter() - t0
    else:
        t_h = rows[-1][2] * 4
    rows.append((n, t_fast, t_h)); print(f"n={n:5d}: tree {t_fast:6.3f} s   Horner {t_h:7.3f} s" + ("  (extrapolated)" if n > 2048 else ""))
ns_, tf, th = map(np.array, zip(*rows))
plt.figure(figsize=(6, 3.3))
plt.loglog(ns_, tf, "o-", label="subproduct tree"); plt.loglog(ns_, th, "s-", label="Horner at each point")
plt.xlabel("n"); plt.ylabel("seconds"); plt.legend(); plt.tight_layout(); plt.show()

Horner's $n^2$ grows with slope $2$ on the log–log plot, the tree with slope just above $1$; in pure Python with `numpy` the crossover comes at a few thousand points. In C libraries such as FLINT or NTL, with better constants for both, the tree wins earlier.

### P9.5 Matrices
Strassen's recursion with a cut-off (below it, `numpy`'s product), and the Schur complement inversion of Theorem 9.9 for symmetric positive definite matrices.

In [ ]:
def strassen(A, B, cut=128):
    n = A.shape[0]
    if n <= cut: return A @ B
    h = n // 2
    a11, a12, a21, a22 = A[:h, :h], A[:h, h:], A[h:, :h], A[h:, h:]
    b11, b12, b21, b22 = B[:h, :h], B[:h, h:], B[h:, :h], B[h:, h:]
    M1 = strassen(a11 + a22, b11 + b22, cut); M2 = strassen(a21 + a22, b11, cut)
    M3 = strassen(a11, b12 - b22, cut); M4 = strassen(a22, b21 - b11, cut)
    M5 = strassen(a11 + a12, b22, cut); M6 = strassen(a21 - a11, b11 + b12, cut)
    M7 = strassen(a12 - a22, b21 + b22, cut)
    return np.block([[M1 + M4 - M5 + M7, M3 + M5], [M2 + M4, M1 - M2 + M3 + M6]])

def spd_inverse(A, mul=lambda X, Y: X @ Y, cut=64):
    n = A.shape[0]
    if n <= cut: return np.linalg.inv(A)
    h = n // 2
    Bm, Ct, Cm, D = A[:h, :h], A[:h, h:], A[h:, :h], A[h:, h:]
    Bi = spd_inverse(Bm, mul, cut)
    CBi = mul(Cm, Bi)
    S = D - mul(CBi, Ct)
    Si = spd_inverse(S, mul, cut)
    SiCBi = mul(Si, CBi)
    top_left = Bi + mul(CBi.T, SiCBi)
    return np.block([[top_left, -SiCBi.T], [-SiCBi, Si]])

n = 1024
X = rng.standard_normal((n, n)); Y = rng.standard_normal((n, n))
t0 = time.perf_counter(); Z1 = X @ Y; t_np = time.perf_counter() - t0
t0 = time.perf_counter(); Z2 = strassen(X, Y); t_st = time.perf_counter() - t0
print(f"Strassen vs numpy: max difference {abs(Z1 - Z2).max():.2e};  time {t_st:.2f} s vs {t_np:.2f} s")
A = X @ X.T + n * np.eye(n)
t0 = time.perf_counter(); Ai = spd_inverse(A); t_sc = time.perf_counter() - t0
t0 = time.perf_counter(); Ai2 = np.linalg.inv(A); t_inv = time.perf_counter() - t0
print(f"Schur inversion: |A A^-1 - I| = {abs(A @ Ai - np.eye(n)).max():.2e}, difference with numpy {abs(Ai - Ai2).max():.2e};  time {t_sc:.2f} s vs {t_inv:.2f} s")

Strassen loses here: `numpy` calls an optimised BLAS whose cubic algorithm is tuned to the memory hierarchy, and Strassen's extra additions and temporary matrices cost more than the one product in eight it saves at these sizes. Its errors are also slightly larger, a known weakness. The Schur recursion is correct and competitive, since most of its time is spent in products.

### P9.6 Recurrences
Fiduccia: $u_N=\sum_jr_ju_j$ with $x^N\bmod P(x)=\sum_jr_jx^j$, computed by repeated squaring with the fast division of P9.3. The companion matrix method uses `numpy` object arrays for exact products modulo $p$, so it is limited to small $d$.

In [ ]:
def poly_powmod(e, Pm):
    # x^e modulo the monic polynomial Pm (coefficients lowest first)
    result = np.array([1], dtype=np.int64); base = np.array([0, 1], dtype=np.int64)
    base = poly_mod(base, Pm)
    for bit in bin(e)[2:]:
        result = poly_mod(pmul(result, result), Pm)
        if bit == "1": result = poly_mod(np.concatenate([[0], result]), Pm)
    return result

def fiduccia(c, u, N):
    d = len(c)
    Pm = np.array([(-ci) % P for ci in reversed(c)] + [1], dtype=np.int64)   # x^d - c1 x^(d-1) - ... - cd
    r = poly_powmod(N, Pm)
    return sum(int(ri) * int(ui) for ri, ui in zip(r, u)) % P

def companion_power(c, u, N):
    d = len(c)
    Mx = np.zeros((d, d), dtype=object)
    Mx[0, :] = c
    for i in range(1, d): Mx[i, i - 1] = 1
    R = np.eye(d, dtype=object); B = Mx.copy(); e = N - (d - 1)
    while e:
        if e & 1: R = R.dot(B) % P
        B = B.dot(B) % P; e >>= 1
    state = np.array(list(reversed(u)), dtype=object)   # (u_{d-1}, ..., u_0)
    return int(R.dot(state)[0] % P)

a_, b_ = 0, 1
for _ in range(90): a_, b_ = b_, a_ + b_
print("F_90 mod p by Fiduccia:", fiduccia([1, 1], [0, 1], 90), "  by iteration:", a_ % P)
N = 10**18
for d in (2, 8, 32, 64):
    c = [random.randrange(P) for _ in range(d)]; u = [random.randrange(P) for _ in range(d)]
    t0 = time.perf_counter(); v1 = fiduccia(c, u, N); t1 = time.perf_counter() - t0
    t0 = time.perf_counter(); v2 = companion_power(c, u, N); t2 = time.perf_counter() - t0
    print(f"d={d:3d}: u_N mod p = {v1:9d}  Fiduccia {t1:6.3f} s, companion matrix {t2:7.3f} s, agree: {v1 == v2}")
for d in (256, 1024):
    c = [random.randrange(P) for _ in range(d)]; u = [random.randrange(P) for _ in range(d)]
    t0 = time.perf_counter(); fiduccia(c, u, N); print(f"d={d:4d}: Fiduccia {time.perf_counter() - t0:.2f} s")

The companion matrix costs $d^3$ per squaring with these exact products, Fiduccia $O(d\log d)$: at $d=64$ the gap is already large, and at $d=1024$ only Fiduccia is practical. The $10^{18}$th term needs about $60$ squarings either way.